# 02. Topic Modeling & Intent Discovery with BERTopic

This notebook applies unsupervised topic discovery to customer support queries using **BERTopic**.
We explore latent semantic clusters, generate interactive/static topic visualizations, and map discovered clusters against true intent labels to evaluate category coherence, taxonomy fragmentation, and lexical overlap.


## 1. Setup & Data Loading

We load the preprocessed raw dataset (`data/raw/bitext_support.csv`) and extract the customer instructions/queries into a list of documents.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic

# Styling
sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (12, 6)

# Locate dataset
csv_path = "../data/raw/bitext_support.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/bitext_support.csv"

df = pd.read_csv(csv_path)
print(f"Total dataset records: {len(df):,}")

# Extract query text and intent labels
docs = df["instruction"].astype(str).tolist()
intents = df["intent"].tolist()

print(f"Loaded {len(docs):,} customer queries ready for topic modeling.")
print(f"Sample query: \"{docs[0]}\" (True Intent: {intents[0]})")


## 2. BERTopic Model Fitting

We initialize BERTopic with:
- `language="english"`
- `calculate_probabilities=True`
- `verbose=True`
- `min_topic_size=50` (ensures stable, dense semantic clusters while suppressing extreme noise)

*(Note: If running on a CPU without GPU acceleration, you can sample 5,000–10,000 queries for faster iteration if needed).*


In [ ]:
# Initialize BERTopic model
# You can adjust min_topic_size to control cluster granularity (default: 50)
topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True,
    min_topic_size=50
)

# Fit model and predict topics for the documents
topics, probs = topic_model.fit_transform(docs)

# Inspect discovered topic information
topic_info = topic_model.get_topic_info()
print(f"Discovered {len(topic_info) - 1} distinct topics (excluding Outliers / Topic -1)")
topic_info.head(15)


## 3. Topic Visualizations

We generate key diagnostic visualizations provided by BERTopic:
1. **Interactive Topic Inter-distance Map** (`visualize_topics`): 2D UMAP projection showing semantic distance between topics.
2. **Top Terms Bar Chart** (`visualize_barchart`): Most representative c-TF-IDF keywords per topic.
3. **Hierarchical Clustering Tree** (`visualize_hierarchy`): Dendrogram illustrating how topics merge at higher abstraction levels.
4. **Similarity Heatmap** (`visualize_heatmap`): Cosine similarity between topic embedding vectors.


In [ ]:
# 1. 2D Inter-topic Distance Map (interactive Plotly map)
fig_topics = topic_model.visualize_topics()
fig_topics.show()


In [ ]:
# 2. Term rank barcharts for the top 15 topics
fig_barchart = topic_model.visualize_barchart(top_n_topics=15)
fig_barchart.show()


In [ ]:
# 3. Hierarchical clustering of discovered topics
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.show()


In [ ]:
# 4. Topic similarity heatmap
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap.show()


## 4. Topic vs Intent Mapping

To evaluate whether unsupervised semantic clusters reflect the ground-truth business taxonomy, we cross-tabulate discovered topics against actual `intent` annotations.


In [ ]:
# Create mapping DataFrame
analysis_df = pd.DataFrame({
    "instruction": docs,
    "true_intent": intents,
    "topic_id": topics
})

# Cross-tabulation table (Topics as rows, Intents as columns)
ctab = pd.crosstab(analysis_df["topic_id"], analysis_df["true_intent"])

# Filter out topic -1 (outliers) for alignment scoring
valid_ctab = ctab[ctab.index != -1]

# Dominant intent per discovered topic
dominant_intents = []
for tid in valid_ctab.index:
    top_intent = valid_ctab.loc[tid].idxmax()
    top_count = valid_ctab.loc[tid].max()
    total_topic_samples = valid_ctab.loc[tid].sum()
    purity = (top_count / total_topic_samples) * 100
    dominant_intents.append({
        "topic_id": tid,
        "dominant_intent": top_intent,
        "sample_count": total_topic_samples,
        "dominant_count": top_count,
        "purity_pct": round(purity, 2)
    })

purity_df = pd.DataFrame(dominant_intents)
print("--- Discovered Topic Purity Summary ---")
print(purity_df.head(15))

# Overall mean topic purity
mean_purity = purity_df["purity_pct"].mean()
print(f"\nAverage Topic Purity: {mean_purity:.2f}% across discovered topics.")


In [ ]:
# Heatmap of Topic-Intent Co-occurrence for top 15 topics
top_topics = purity_df.sort_values(by="sample_count", ascending=False)["topic_id"].head(15)
subset_ctab = valid_ctab.loc[top_topics]

# Keep intents that have noticeable activity
active_intents = subset_ctab.columns[subset_ctab.sum(axis=0) > 20]
plot_matrix = subset_ctab[active_intents]

plt.figure(figsize=(16, 9))
sns.heatmap(plot_matrix, cmap="Blues", annot=True, fmt="d", cbar=True)
plt.title("Topic vs. Intent Alignment Matrix (Top 15 Topics)", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Ground Truth Intent", labelpad=10)
plt.ylabel("Discovered Topic ID", labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5. Observations & Qualitative Analysis

Fill in the sections below based on the actual outputs from running the cells above:

### 1. Discovered Topics vs 27 Known Intents
- **Number of Discovered Topics:** *(State how many valid topics BERTopic produced with min_topic_size=50 vs the 27 true intents)*
- **Outlier Proportion (Topic -1):** *(Percentage of queries classified as noise)*

### 2. Clean One-to-One Intent Mappings
- *(Which intents mapped cleanly to a single high-purity topic? E.g., `cancel_order`, `track_order`, etc.)*

### 3. Fragmented Intents (One Intent split into Multiple Topics)
- *(Which intents were subdivided into multiple fine-grained subtopics? E.g., queries asking about shipping delays vs tracking numbers)*

### 4. Overlapping & Blended Clusters
- *(Which topics contained a mix of two or more intents? What common vocabulary caused the confusion?)*

### 5. Taxonomy Recommendations
- *(Does the unsupervised clustering suggest any intents that could be merged or hierarchically structured in a production customer support system?)*
